# 04 - SARIMAX model
Part 4: stationarity, AIC grid search over (p,d,q), fit, forecast with confidence intervals, residual diagnostics.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from appliance_energy import config, data, features, plotting
from appliance_energy.models import sarimax as sarimax_model

In [ ]:
df = data.load_appliance_data()
y = df[config.TARGET]
train = y.iloc[:-config.TEST_STEPS]
test = y.iloc[-config.TEST_STEPS:]

In [ ]:
exog = features.get_exog_columns(df)
X_train = exog.iloc[:-config.TEST_STEPS]
X_test = exog.iloc[-config.TEST_STEPS:]

## AIC grid search
Warning: `quick=False` runs the full 7x3x7 grid from the assignment spec and is slow. Start with `quick=True`.

In [ ]:
grid = sarimax_model.grid_search_sarimax(train, X_train=X_train, quick=True)
grid.head(10)

In [ ]:
best_p, best_d, best_q = grid.dropna(subset=['aic']).iloc[0][['p','d','q']].astype(int)
fit = sarimax_model.fit_sarimax(train, order=(best_p, best_d, best_q), X_train=X_train)
fit.summary()

In [ ]:
mean, lower, upper = sarimax_model.forecast_sarimax(fit, horizon=len(test), index=test.index, X_test=X_test)
plotting.plot_forecast_with_ci(test, mean, lower, upper)

In [ ]:
plotting.plot_residual_diagnostics(fit.resid)

In [ ]:
sarimax_model.residual_diagnostics_summary(fit)